In [1]:
import pandas as pd
import os
import pyarrow

In [2]:
# Configuração de diretórios e nomes dos arquivos
diretorio_dados = r"C:\VSCodeWorkspace\TCC_Final\data"
arquivo_parquet_original = 'lista_economatica_dados_Jan_2010_Dezembro_2025.parquet'
arquivo_ativos_alvo = 'diagnostico_econometrico_130_ativos.csv'

arquivo_saida_parquet = 'lista_economatica_dados_filtrados.parquet'
arquivo_saida_csv = 'lista_economatica_dados_filtrados.csv'

In [3]:
# 1. Leitura dos ativos alvo do diagnóstico econométrico
caminho_diagnostico = os.path.join(diretorio_dados, arquivo_ativos_alvo)
print(f"Lendo lista de ativos de: {caminho_diagnostico}")

df_diagnostico = pd.read_csv(caminho_diagnostico)
# Limpando underscores espúrios (como WEGE3_ -> WEGE3)
tickers_alvo = df_diagnostico['Ativo'].str.rstrip('_').tolist()

print(f"[OK] Total de ativos identificados no diagnóstico: {len(tickers_alvo)}")
print(f"Primeiros 5 ativos: {tickers_alvo[:5]}")
print(f"Últimos 5 ativos: {tickers_alvo[-5:]}")

Lendo lista de ativos de: C:\VSCodeWorkspace\TCC_Final\data\diagnostico_econometrico_130_ativos.csv
[OK] Total de ativos identificados no diagnóstico: 130
Primeiros 5 ativos: ['ABCB4', 'ABEV3', 'AGRO3', 'ALPA4', 'AMAR3']
Últimos 5 ativos: ['VIVT3', 'VLID3', 'VSTE3', 'WEGE3', 'YDUQ3']


In [4]:
# 2. Carregamento do dataset original Parquet
caminho_parquet = os.path.join(diretorio_dados, arquivo_parquet_original)
print(f"Lendo arquivo Parquet original de: {caminho_parquet}")

df_original = pd.read_parquet(caminho_parquet, engine='pyarrow')
print(f"[OK] Leitura concluída! Shape original: {df_original.shape}")

Lendo arquivo Parquet original de: C:\VSCodeWorkspace\TCC_Final\data\lista_economatica_dados_Jan_2010_Dezembro_2025.parquet
[OK] Leitura concluída! Shape original: (3967, 497)


In [5]:
# 3. Filtragem das colunas (mantendo 'Data' e os tickers correspondentes)
colunas_disponiveis = df_original.columns.tolist()
ativos_encontrados = [t for t in tickers_alvo if t in colunas_disponiveis]
ativos_faltantes = [t for t in tickers_alvo if t not in colunas_disponiveis]

print(f"Ativos encontrados na base de dados: {len(ativos_encontrados)} / {len(tickers_alvo)}")
if ativos_faltantes:
    print(f"[AVISO] Atenção: {len(ativos_faltantes)} ativos não foram encontrados na base Parquet: {ativos_faltantes}")
else:
    print("[OK] Excelente! Todos os 130 ativos estão presentes no arquivo Parquet original.")

# Garantir que a coluna 'Data' seja a primeira
colunas_filtradas = ['Data'] + ativos_encontrados
df_filtrado = df_original[colunas_filtradas]

print(f"\nNova base de dados filtrada:")
print(f"- Shape: {df_filtrado.shape}")
print(f"- Colunas: {len(df_filtrado.columns)} (incluindo 'Data')")

Ativos encontrados na base de dados: 130 / 130
[OK] Excelente! Todos os 130 ativos estão presentes no arquivo Parquet original.

Nova base de dados filtrada:
- Shape: (3967, 131)
- Colunas: 131 (incluindo 'Data')


In [6]:
# 4. Salvando os arquivos resultantes em Parquet e CSV
caminho_saida_parquet = os.path.join(diretorio_dados, arquivo_saida_parquet)
caminho_saida_csv = os.path.join(diretorio_dados, arquivo_saida_csv)

print(f"Salvando arquivo Parquet em: {caminho_saida_parquet}")
df_filtrado.to_parquet(caminho_saida_parquet, index=False, engine='pyarrow')
print("[OK] Parquet salvo!")

print(f"\nSalvando arquivo CSV em: {caminho_saida_csv}")
df_filtrado.to_csv(caminho_saida_csv, index=False, sep=';', decimal=',')
print("[OK] CSV salvo!")

Salvando arquivo Parquet em: C:\VSCodeWorkspace\TCC_Final\data\lista_economatica_dados_filtrados.parquet
[OK] Parquet salvo!

Salvando arquivo CSV em: C:\VSCodeWorkspace\TCC_Final\data\lista_economatica_dados_filtrados.csv


[OK] CSV salvo!


In [7]:
# 5. Validação Rápida da Base Gerada
print("=== Validação Rápida dos Resultados ===")
print(f"Shape final: {df_filtrado.shape}")
print(f"Número de ativos (excluindo Data): {len(df_filtrado.columns) - 1}")
print(f"Período temporal: de {df_filtrado['Data'].iloc[0]} a {df_filtrado['Data'].iloc[-1]}")

# Exibindo o head
df_filtrado.head()

=== Validação Rápida dos Resultados ===
Shape final: (3967, 131)
Número de ativos (excluindo Data): 130
Período temporal: de 04/01/2010 a 30/12/2025


,Data,ABCB4,ABEV3,AGRO3,ALPA4,AMAR3,AMER3,AXIA3,AXIA6,B3SA3,...,UNIP6,USIM3,USIM5,VALE3,VIVR3,VIVT3,VLID3,VSTE3,WEGE3,YDUQ3
0,04/01/2010,3.828227,3.181294,4.531978,2.086913,38.729817,3863.522415,12.119488,9.262757,2.244572,...,1.785257,19.622723,18.808727,18.739653,10547.94700,5.563114,7.582252,103.027658,1.885304,5.076226
1,05/01/2010,3.930519,3.200008,NaN,2.094172,40.238771,3771.316225,12.020069,9.122413,2.253239,...,1.926198,19.745822,18.864047,18.914348,10487.15192,5.542499,7.574395,101.148735,1.881190,5.137032
2,06/01/2010,3.995615,3.232652,NaN,2.088728,40.205238,3812.850545,11.866130,9.007330,2.338168,...,1.894878,19.845839,18.845607,19.314690,11155.89784,5.448259,7.601895,100.522427,1.891475,5.220902
3,07/01/2010,3.936719,3.247831,4.660828,2.110504,41.177675,3856.046237,12.141937,9.206619,2.393633,...,1.832238,19.753515,19.022630,19.394759,11247.09046,5.448259,7.621538,100.209273,1.923360,5.137032
4,08/01/2010,3.905721,3.241593,NaN,2.132281,43.189614,3763.840048,12.347189,9.313282,2.338168,...,1.816577,20.007407,18.753407,19.584011,11186.29538,5.443841,7.778683,100.052696,1.993300,5.137032
